# Hybrid Recommendation System

## Project Overview

This notebook implements a Hybrid Recommendation System by combining multiple recommendation techniques developed throughout this project.

The goal is to leverage the strengths of different recommendation algorithms while minimizing their individual limitations.

## Recommendation Techniques Used

1. Popularity-Based Recommendation
2. User-Based Collaborative Filtering
3. Item-Based Collaborative Filtering
4. Matrix Factorization (SVD)
5. FAISS Vector Search
6. Neural Collaborative Filtering (NCF)

## Why Hybrid Recommendation?

Each recommendation algorithm performs well under different scenarios.

- Popularity-based methods work well for new users.
- Collaborative Filtering captures user preferences.
- Matrix Factorization learns hidden user-item relationships.
- FAISS enables efficient nearest-neighbor retrieval.
- Neural Collaborative Filtering models complex interactions.

Combining these methods produces more accurate, robust, and personalized recommendations.

## Workflow

1. Load all saved models and embeddings
2. Load the MovieLens dataset
3. Generate recommendations from multiple models
4. Combine recommendation scores
5. Rank movies
6. Display Top-N personalized recommendations

By the end of this notebook, a complete production-style Hybrid Recommendation System will be implemented.

In [1]:
# ==========================================
# Import Libraries
# ==========================================

import pandas as pd
import numpy as np

import faiss
import torch

print("Libraries Loaded Successfully!")

Libraries Loaded Successfully!


In [2]:
# ==========================================
# Load Saved Files
# ==========================================

user_embeddings = pd.read_csv("models/user_embeddings.csv")

item_embeddings = pd.read_csv("models/item_embeddings.csv")

user_mapping = pd.read_csv("models/user_mapping.csv")

item_mapping = pd.read_csv("models/item_mapping.csv")

faiss_index = faiss.read_index("models/user_faiss.index")

print("Saved Models Loaded Successfully!")

Saved Models Loaded Successfully!


In [3]:
# ==========================================
# Load Dataset
# ==========================================

ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t",
    names=["user_id","movie_id","rating","timestamp"]
)

movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0,1],
    names=["movie_id","movie_title"]
)

movie_ratings = ratings.merge(
    movies,
    on="movie_id"
)

print(movie_ratings.head())

   user_id  movie_id  rating  timestamp                 movie_title
0      196       242       3  881250949                Kolya (1996)
1      186       302       3  891717742    L.A. Confidential (1997)
2       22       377       1  878887116         Heavyweights (1994)
3      244        51       2  880606923  Legends of the Fall (1994)
4      166       346       1  886397596         Jackie Brown (1997)


# Neural Collaborative Filtering Recommendation System

## Objective

In this notebook we use the trained Neural Collaborative Filtering (NCF) model
to generate personalized movie recommendations.

Unlike traditional collaborative filtering, the trained neural network learns
non-linear interactions between users and movies.

The workflow is:

1. Load trained NCF model
2. Load learned user embeddings
3. Load learned movie embeddings
4. Predict ratings for unseen movies
5. Rank movies
6. Recommend Top-N movies

This notebook represents the final recommendation engine of our project.

In [4]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

print("Libraries Loaded Successfully!")

Libraries Loaded Successfully!


In [5]:
class NCF(nn.Module):

    def __init__(self,
                 num_users,
                 num_movies,
                 embedding_dim=50):

        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.movie_embedding = nn.Embedding(
            num_movies,
            embedding_dim
        )

        self.fc1 = nn.Linear(100,128)
        self.fc2 = nn.Linear(128,64)
        self.output = nn.Linear(64,1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self,user,movie):

        user_vec = self.user_embedding(user)
        movie_vec = self.movie_embedding(movie)

        x = torch.cat([user_vec,movie_vec],dim=1)

        x = self.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.relu(self.fc2(x))

        rating = self.output(x)

        return rating.squeeze()

In [6]:
model = NCF(
    num_users=943,
    num_movies=1682,
    embedding_dim=50
)

model.load_state_dict(
    torch.load(
        "models/ncf_model.pth",
        map_location=torch.device("cpu")
    )
)

model.eval()

print("NCF Model Loaded Successfully!")

NCF Model Loaded Successfully!


In [8]:
# ==========================================
# Load Dataset
# ==========================================

ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t",
    names=[
        "user_id",
        "movie_id",
        "rating",
        "timestamp"
    ]
)

movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0,1],
    names=[
        "movie_id",
        "movie_title"
    ]
)

ratings = ratings.merge(
    movies,
    on="movie_id"
)

print("Dataset Loaded Successfully!")

ratings.head()

Dataset Loaded Successfully!


,user_id,movie_id,rating,timestamp,movie_title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


# Generate Personalized Movie Recommendations

In this section, we use the trained Neural Collaborative Filtering (NCF) model
to predict ratings for movies that the target user has not watched.

The recommendation process consists of:

1. Select a target user.
2. Identify movies already watched.
3. Find unseen movies.
4. Predict ratings for unseen movies.
5. Rank the predicted ratings.
6. Recommend the Top-N movies.

Unlike popularity-based recommendation, these recommendations are personalized
according to the learned user and movie embeddings.

In [9]:
# ==========================================
# Select User
# ==========================================

target_user = 1

print("Target User:", target_user)

Target User: 1


In [10]:
# ==========================================
# Movies Already Watched
# ==========================================

watched_movies = ratings[
    ratings["user_id"] == target_user
]["movie_id"].tolist()

print("Movies Already Watched :", len(watched_movies))

ratings[
    ratings["user_id"] == target_user
][["movie_title","rating"]].head(10)

Movies Already Watched : 272


,movie_title,rating
202,Three Colors: White (1994),4
305,"Grand Day Out, A (1992)",3
333,Desperado (1995),4
334,Glengarry Glen Ross (1992),4
478,Angels and Insects (1995),4
639,Groundhog Day (1993),5
687,Delicatessen (1991),5
820,"Hunt for Red October, The (1990)",4
933,Dirty Dancing (1987),2
972,"Rock, The (1996)",3


In [11]:
# ==========================================
# Find Unseen Movies
# ==========================================

all_movies = ratings["movie_id"].unique()

unseen_movies = [
    movie
    for movie in all_movies
    if movie not in watched_movies
]

print("Total Movies :", len(all_movies))
print("Unseen Movies:", len(unseen_movies))

Total Movies : 1682
Unseen Movies: 1410


# Predict Ratings for Unseen Movies

The trained NCF model estimates how much the target user would like each unseen movie.

For every unseen movie:

1. Create a (User, Movie) pair.
2. Pass the pair through the trained neural network.
3. Predict the user's rating.
4. Store the predicted rating.

After predicting ratings for all unseen movies, we rank them in descending order to generate personalized recommendations.

In [12]:
# ==========================================
# Create Movie Encoding Mapping
# ==========================================

movie_mapping = (
    ratings[["movie_id", "movie_title"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

movie_mapping["movie_encoded"] = movie_mapping.index

movie_mapping.head()

,movie_id,movie_title,movie_encoded
0,242,Kolya (1996),0
1,302,L.A. Confidential (1997),1
2,377,Heavyweights (1994),2
3,51,Legends of the Fall (1994),3
4,346,Jackie Brown (1997),4


In [13]:
# ==========================================
# Create Movie Lookup Dictionary
# ==========================================

movie_encoder = dict(
    zip(
        movie_mapping.movie_id,
        movie_mapping.movie_encoded
    )
)

movie_decoder = dict(
    zip(
        movie_mapping.movie_encoded,
        movie_mapping.movie_title
    )
)

print("Total Movies:", len(movie_encoder))

Total Movies: 1682


In [14]:
# ==========================================
# Predict Ratings
# ==========================================

predictions = []

user_tensor = torch.tensor([target_user - 1])

for movie in unseen_movies:

    encoded_movie = movie_encoder[movie]

    movie_tensor = torch.tensor([encoded_movie])

    with torch.no_grad():

        predicted_rating = model(
            user_tensor,
            movie_tensor
        ).item()

    predictions.append(
        (
            movie,
            movie_decoder[encoded_movie],
            predicted_rating
        )
    )

print("Total Predictions:", len(predictions))

Total Predictions: 1410


# Rank Predicted Movies

The NCF model has predicted ratings for every movie the user has not watched.

In this step we:

1. Sort all predicted ratings in descending order.
2. Select the highest-rated movies.
3. Display the Top-10 personalized recommendations.

These movies represent the recommendations generated by the trained Neural Collaborative Filtering model.

In [16]:
# ==========================================
# Sort Predictions
# ==========================================

recommendations = pd.DataFrame(
    predictions,
    columns=[
        "movie_id",
        "movie_title",
        "predicted_rating"
    ]
)

recommendations = recommendations.sort_values(
    by="predicted_rating",
    ascending=False
)

recommendations.head(10)

,movie_id,movie_title,predicted_rating
106,408,"Close Shave, A (1995)",4.877297
3,474,Dr. Strangelove or: How I Learned to Stop Worr...,4.832566
18,603,Rear Window (1954),4.689851
690,867,"Whole Wide World, The (1996)",4.567297
35,427,To Kill a Mockingbird (1962),4.484468
987,1449,Pather Panchali (1955),4.462897
408,510,"Magnificent Seven, The (1954)",4.462520
103,483,Casablanca (1942),4.450676
446,657,"Manchurian Candidate, The (1962)",4.432461
1347,1467,"Saint of Fort Washington, The (1993)",4.424121


In [17]:
# ==========================================
# Top 10 Recommendations
# ==========================================

top10 = recommendations.head(10).reset_index(drop=True)

print("Top 10 Recommended Movies")
print("-" * 50)

top10

Top 10 Recommended Movies
--------------------------------------------------


,movie_id,movie_title,predicted_rating
0,408,"Close Shave, A (1995)",4.877297
1,474,Dr. Strangelove or: How I Learned to Stop Worr...,4.832566
2,603,Rear Window (1954),4.689851
3,867,"Whole Wide World, The (1996)",4.567297
4,427,To Kill a Mockingbird (1962),4.484468
5,1449,Pather Panchali (1955),4.462897
6,510,"Magnificent Seven, The (1954)",4.462520
7,483,Casablanca (1942),4.450676
8,657,"Manchurian Candidate, The (1962)",4.432461
9,1467,"Saint of Fort Washington, The (1993)",4.424121


In [18]:
# ==========================================
# Print Recommendations
# ==========================================

for i, row in top10.iterrows():

    print(f"{i+1}. {row['movie_title']}")
    print(f"Predicted Rating : {row['predicted_rating']:.3f}")
    print("-" * 50)

1. Close Shave, A (1995)
Predicted Rating : 4.877
--------------------------------------------------
2. Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963)
Predicted Rating : 4.833
--------------------------------------------------
3. Rear Window (1954)
Predicted Rating : 4.690
--------------------------------------------------
4. Whole Wide World, The (1996)
Predicted Rating : 4.567
--------------------------------------------------
5. To Kill a Mockingbird (1962)
Predicted Rating : 4.484
--------------------------------------------------
6. Pather Panchali (1955)
Predicted Rating : 4.463
--------------------------------------------------
7. Magnificent Seven, The (1954)
Predicted Rating : 4.463
--------------------------------------------------
8. Casablanca (1942)
Predicted Rating : 4.451
--------------------------------------------------
9. Manchurian Candidate, The (1962)
Predicted Rating : 4.432
--------------------------------------------------
10. Saint 